# Detecção e Segmentação de Objetos com RF-DETR (Roboflow)

Neste notebook, iremos explorar o `RF-DETR`, um modelo de detecção e segmentação de objetos baseado em transformers, desenvolvido pela Roboflow como alternativa em tempo real aos modelos da família YOLO.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93 
    !pip install opencv-contrib-python==5.0.0.93
    !pip install rfdetr
    !pip install moviepy==2.2.1
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que já usamos em outros notebooks, vamos utilizar mais duas.
* `rfdetr`: Biblioteca da Roboflow com os modelos RF-DETR de detecção e segmentação de objetos.
* `supervision`: Biblioteca da Roboflow para processar e visualizar resultados de detecção/segmentação de forma padronizada (funciona com RF-DETR, YOLO e diversos outros modelos).
* `moviepy`: Biblioteca para editar e apresentar vídeos.

In [ ]:
from rfdetr import RFDETRNano, RFDETRSegNano
import supervision as sv

from pathlib import Path
from moviepy import VideoFileClip

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image, Video

## 1. Detecção de Objetos

Para a detecção de objetos, vamos usar o `RFDETRNano`, a versão mais leve e rápida do RF-DETR.

### 1.1. Carregando o modelo

A biblioteca `rfdetr` facilita o acesso ao modelo, baixando e carregando os pesos, já treinados na base COCO (as mesmas 80 classes usadas pelo YOLO).

In [ ]:
detect_model = RFDETRNano()

### 1.2. Carregando imagem

Vamos usar a mesma imagem do notebook anterior para facilitar a comparação.

In [ ]:
img = cv2.imread('imagens/02/futebol.webp')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

### 1.3. Inferência do Modelo

Assim como a `ultralytics`, a biblioteca `rfdetr` também facilita a inferência: não precisamos converter a imagem em blob, bastando garantir que ela esteja no formato RGB.

In [ ]:
detections = detect_model.predict(img_rgb, threshold=0.5)

A variável `detections` é um objeto `Detections` da biblioteca `supervision`, que reúne as caixas, classes e confianças de todos os objetos detectados na imagem.

### 1.4. Apresentando e Interpretando Resultados

Em modelos de detecção de objetos, usaremos basicamente as propriedades `xyxy`, `confidence` e `data['class_name']` do objeto `Detections`.

In [ ]:
for xyxy, confianca, classe in zip(detections.xyxy, detections.confidence, detections.data['class_name']):
    print(
        classe,
        round(float(confianca), 2),
        xyxy.round().tolist()
    )

Outra coisa que a biblioteca `supervision` facilita é a apresentação dos resultados, através dos seus "annotators".

In [ ]:
labels = [
    f"{classe} {confianca:.2f}"
    for classe, confianca in zip(detections.data['class_name'], detections.confidence)
]

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

result_img = box_annotator.annotate(scene=img_rgb.copy(), detections=detections)
result_img = label_annotator.annotate(scene=result_img, detections=detections, labels=labels)

plt.figure(figsize=(12, 8))
plt.imshow(result_img)
plt.axis('off')
plt.show()

## 2. Segmentação de Objetos

Vamos usar o `RFDETRSegNano`, a versão de segmentação do RF-DETR.

### 2.1. Carregando o modelo

In [ ]:
seg_model = RFDETRSegNano()

### 2.2. Inferência do Modelo

Vamos usar a mesma imagem para facilitar a comparação.

In [ ]:
detections = seg_model.predict(img_rgb, threshold=0.5)

### 2.3. Apresentando e Interpretando Resultados

Agora `detections` também traz a propriedade `mask`, com a máscara de segmentação de cada objeto identificado.

In [ ]:
mask = detections.mask[0]

plt.imshow(mask, cmap='gray')
plt.title("Máscara do Objeto 1")
plt.axis('off')
plt.show()

#### 2.3.1. Sobrepondo máscaras

Diferente do exemplo anterior com YOLO, aqui vamos aproveitar os próprios "annotators" da `supervision` para sobrepor caixas, rótulos e máscaras, sem precisar escrever manualmente a lógica de sobreposição.

In [ ]:
mask_annotator = sv.MaskAnnotator()
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

labels = [
    f"{classe} {confianca:.2f}"
    for classe, confianca in zip(detections.data['class_name'], detections.confidence)
]

result_img = mask_annotator.annotate(scene=img_rgb.copy(), detections=detections)
result_img = box_annotator.annotate(scene=result_img, detections=detections)
result_img = label_annotator.annotate(scene=result_img, detections=detections, labels=labels)

plt.figure(figsize=(12, 8))
plt.imshow(result_img)
plt.axis('off')
plt.show()

## 3. Segmentação de Pessoas em Vídeo

O RF-DETR também pode ser aplicado quadro a quadro em vídeos. Vamos processar o vídeo `pescadores.mp4`, segmentando apenas as pessoas presentes em cada quadro e salvando o resultado em um novo vídeo.

### 3.1. Carregando o vídeo

Usamos o `cv2.VideoCapture` para abrir o vídeo e ler suas propriedades (resolução, taxa de quadros e número total de quadros).

In [ ]:
video_path = 'imagens/02/pescadores.mp4'
output_path = 'output/pescadores_segmentado.mp4'

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_quadros = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Resolução: {largura}x{altura}, {fps:.1f} fps, {total_quadros} quadros")

### 3.2. Processando o vídeo, quadro a quadro

Para cada quadro do vídeo, seguimos o mesmo processo usado nas imagens: convertemos para RGB, executamos a inferência do `seg_model` e sobrepomos as máscaras encontradas. A diferença é que, aqui, filtramos as detecções para manter apenas os objetos da classe `person`, usando `detections.data['class_name']`.

Cada quadro anotado é gravado em um novo arquivo de vídeo com o `cv2.VideoWriter`.

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (largura, altura))

mask_annotator = sv.MaskAnnotator()

quadros_amostra = []
intervalo_amostra = max(total_quadros // 4, 1)

quadro_idx = 0
while True:
    ok, quadro = cap.read()
    if not ok:
        break

    quadro_rgb = cv2.cvtColor(quadro, cv2.COLOR_BGR2RGB)
    detections = seg_model.predict(quadro_rgb, threshold=0.5)
    pessoas = detections[detections.data['class_name'] == 'person']

    quadro_anotado = mask_annotator.annotate(scene=quadro_rgb.copy(), detections=pessoas)
    writer.write(cv2.cvtColor(quadro_anotado, cv2.COLOR_RGB2BGR))

    if quadro_idx % intervalo_amostra == 0:
        quadros_amostra.append(quadro_anotado)

    quadro_idx += 1

cap.release()
writer.release()

print(f"Vídeo processado salvo em '{output_path}' ({quadro_idx} quadros).")

### 3.3. Apresentando o Resultado

Como o `opencv-python` no Windows não vem com um codificador H.264, o vídeo gerado usa o codec MPEG-4 (`mp4v`). Ele é válido e pode ser aberto normalmente em players como o VLC, mas alguns navegadores não conseguem reproduzi-lo embutido no notebook. Por isso, vamos primeiro conferir uma amostra dos quadros processados com o `matplotlib`, e depois tentar exibir o vídeo completo.

In [ ]:
fig, eixos = plt.subplots(1, len(quadros_amostra), figsize=(5 * len(quadros_amostra), 5))

for eixo, quadro in zip(eixos, quadros_amostra):
    eixo.imshow(quadro)
    eixo.axis('off')

plt.tight_layout()
plt.show()

Se o seu navegador suportar o codec do vídeo gerado, ele será reproduzido logo abaixo. Caso contrário, abra o arquivo `output/pescadores_segmentado.mp4` diretamente em um player de vídeo.

In [ ]:
def display_video(filename: str, **kwargs) -> None:
    """ Apresenta vídeo no notebook """
    clip = VideoFileClip(filename)
    html_embed = clip.display_in_notebook(**kwargs)
    # O moviepy cria um arquivo temporário. Só vamos apagar ele.
    Path("__temp__.mp4").unlink(missing_ok=True)
    return html_embed

In [ ]:
display_video(output_path, loop=1)